In [1]:
import pandas as pd
import numpy as np
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# --- Step 1: Load Data with Dummies ---
try:
    # Use the file previously created/uploaded
    df_with_dummies = pd.read_csv("DC_Master_Data_With_Dummies.csv", parse_dates=['Date'], index_col='Date')
    # Ensure the frequency is set correctly
    df_with_dummies = df_with_dummies.asfreq('QS-JAN')
    print("Data with dummies loaded successfully.")

    # --- Step 2: Prepare Target and Base Predictors ---
    print("Preparing stationary variables and lags...")

    # Target variable (stationary)
    target = df_with_dummies['House_Index'].diff(1).rename('Target_House_Index_diff1')

    # Create stationary versions of predictors (as needed for lagging)
    predictors_stationary = pd.DataFrame(index=df_with_dummies.index)
    predictors_stationary['Unemployment_Rate'] = df_with_dummies['Unemployment_Rate'] # I(0)
    predictors_stationary['CPI_diff1'] = df_with_dummies['CPI'].diff(1)
    predictors_stationary['Poverty_Rate_diff1'] = df_with_dummies['Poverty_Rate'].diff(1)
    predictors_stationary['Median_Household_Income_diff1'] = df_with_dummies['Median_Household_Income'].diff(1)
    predictors_stationary['Interest_Rate_diff1'] = df_with_dummies['Interest_Rate'].diff(1)
    predictors_stationary['Mortgage_Rate_diff1'] = df_with_dummies['Mortgage_Rate'].diff(1)
    predictors_stationary['Population_diff2'] = df_with_dummies['Population'].diff(2)
    predictors_stationary['GDP_diff3'] = df_with_dummies['GDP'].diff(3)

    # --- Step 3: Create Predictor Matrix (X) with Dummies and Lags ---
    X_lasso = df_with_dummies[['Recession', 'Hot_Market']].copy() # Start with dummies

    max_lag = 4
    predictor_cols = predictors_stationary.columns # Get list of stationary predictors

    # Add lags 1 through max_lag for each stationary predictor
    for col in predictor_cols:
        for lag in range(1, max_lag + 1):
            X_lasso[f'{col}_L{lag}'] = predictors_stationary[col].shift(lag)

    # --- Step 4: Combine Target and Predictors, Drop NaNs ---
    df_lasso_input = pd.concat([target, X_lasso], axis=1)

    # Drop rows with NaNs (introduced by differencing in target/predictors AND lagging)
    # Max lag is 4, max initial diff for predictors was 3. Need to drop at least 4 rows.
    initial_len = len(df_lasso_input)
    df_lasso_input.dropna(inplace=True)
    final_len = len(df_lasso_input)
    print(f"Combined DataFrame created. Dropped {initial_len - final_len} rows due to NaNs.")
    print(f"Final shape for LASSO input: {df_lasso_input.shape}")


    # --- Step 5: Save the Prepared Data ---
    output_file_lasso = 'DC_Data_For_Lasso.csv'
    # Use reset_index() to include the 'Date' column in the CSV
    df_lasso_input.reset_index().to_csv(output_file_lasso, index=False)
    print(f"\nData prepared for LASSO/Regression saved to {output_file_lasso}")

    # --- Step 6: Display Sample of Prepared Data ---
    print("\n--- Head of data prepared for LASSO ---")
    print(df_lasso_input.head())
    print("\n--- Tail of data prepared for LASSO ---")
    print(df_lasso_input.tail())


except FileNotFoundError:
    print("Error: The file 'DC_Master_Data_With_Dummies.csv' was not found.")
except KeyError as e:
    print(f"Error: A required column was not found: {e}")
    print("Please check column names.")
except Exception as e:
    print(f"An error occurred during analysis: {e}")

Data with dummies loaded successfully.
Preparing stationary variables and lags...
Combined DataFrame created. Dropped 7 rows due to NaNs.
Final shape for LASSO input: (133, 35)

Data prepared for LASSO/Regression saved to DC_Data_For_Lasso.csv

--- Head of data prepared for LASSO ---
            Target_House_Index_diff1  Recession  Hot_Market  \
Date                                                          
1991-10-01                      1.60          0           0   
1992-01-01                      0.31          0           0   
1992-04-01                     -0.82          0           0   
1992-07-01                      0.82          0           0   
1992-10-01                      0.11          0           0   

            Unemployment_Rate_L1  Unemployment_Rate_L2  Unemployment_Rate_L3  \
Date                                                                           
1991-10-01                   6.1                   5.2                   5.3   
1992-01-01                   6.2 